# Lesson 13: 音を周波数で見る

**コンパニオンノートブック** — 詳しい解説は本文 Lesson 13 を参照してください。

## セットアップ

In [ ]:
# --- 最初に1回だけ実行 ---
import sys
try:
    import google.colab
    !pip install -q japanize-matplotlib
    !git clone -q https://github.com/ggszk/simple-sound-programming.git
    sys.path.append('/content/simple-sound-programming')
except ImportError:
    sys.path.append('..')

from audio_lib.notebook import setup_environment
setup_environment()

## このレッスンで学ぶこと

- FFT（高速フーリエ変換）の直感的な意味を理解する
- スペクトルの読み方を学び、音色の違いを周波数の視点で説明できるようになる
- スペクトログラムで「時間とともに変化する周波数」を可視化する
- 楽器音や合成音の周波数分析を実践する
- 窓関数がなぜ必要かを直感的に理解する


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from audio_lib import (
    sine_wave, sawtooth_wave, square_wave, triangle_wave,
    additive_synth, adsr, AudioSignal,
)
from audio_lib.notebook import play_sound, plot_waveform, plot_spectrum

## 13.1 加算合成の「逆」を考える

In [ ]:
f0 = 440
sig = additive_synth(f0, {1: 1.0, 2: 0.5, 3: 0.33, 4: 0.25, 5: 0.2})
display(play_sound(sig, "加算合成（5つの倍音）"))

### NumPy で FFT を実行する

In [ ]:
# テスト用: 440Hz + 880Hz + 1320Hz の合成音
sample_rate = 44100
duration = 1.0
t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

data = np.sin(2 * np.pi * 440 * t) + 0.5 * np.sin(2 * np.pi * 880 * t) + 0.3 * np.sin(2 * np.pi * 1320 * t)

# FFT を実行
fft_result = np.fft.fft(data)
fft_freq = np.fft.fftfreq(len(data), 1 / sample_rate)
fft_magnitude = np.abs(fft_result)

print(f"入力データ: {len(data)} サンプル")
print(f"FFT 結果: {len(fft_result)} 個の複素数")
print(f"周波数軸: {len(fft_freq)} 個")

## 13.3 スペクトルを読む

In [ ]:
# 正の周波数だけを取り出す
positive = fft_freq >= 0
freq = fft_freq[positive]
magnitude = fft_magnitude[positive]

# 表示範囲を 0〜3000Hz に絞る
range_idx = freq <= 3000

plt.figure(figsize=(12, 4))
plt.plot(freq[range_idx], magnitude[range_idx])
plt.title("合成音のスペクトル（440Hz + 880Hz + 1320Hz）")
plt.xlabel("周波数 (Hz)")
plt.ylabel("振幅")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 13.4 dB スケール — 大きさの違いを見やすくする

In [ ]:
# 同じスペクトルを dB スケールで表示
magnitude_db = 20 * np.log10(magnitude[range_idx] + 1e-10)  # 0 割り防止

plt.figure(figsize=(12, 4))
plt.plot(freq[range_idx], magnitude_db)
plt.title("合成音のスペクトル（dBスケール）")
plt.xlabel("周波数 (Hz)")
plt.ylabel("振幅 (dB)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 13.5 いろいろな波形のスペクトルを比較する

In [ ]:
f0 = 440
dur = 1.0

waves = {
    "サイン波": sine_wave(f0, dur),
    "ノコギリ波": sawtooth_wave(f0, dur),
    "矩形波": square_wave(f0, dur),
    "三角波": triangle_wave(f0, dur),
}

fig, axes = plt.subplots(4, 1, figsize=(12, 12), sharex=True)

for ax, (name, sig) in zip(axes, waves.items()):
    data = sig.data
    fft_mag = np.abs(np.fft.fft(data))
    freqs = np.fft.fftfreq(len(data), 1 / sig.sample_rate)
    pos = freqs >= 0
    f = freqs[pos]
    m = fft_mag[pos]
    rng = f <= 5000

    ax.plot(f[rng], 20 * np.log10(m[rng] + 1e-10))
    ax.set_ylabel(f"{name}\n振幅 (dB)")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("周波数 (Hz)")
fig.suptitle("4つの基本波形のスペクトル比較", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 聞き比べ
for name, sig in waves.items():
    display(play_sound(sig, name))

### 問題を見てみよう

In [ ]:
# 440Hz（きれいに収まる）と 443Hz（収まらない）を比較
dur = 0.1  # 短い区間
t_short = np.linspace(0, dur, int(sample_rate * dur), endpoint=False)

data_440 = np.sin(2 * np.pi * 440 * t_short)
data_443 = np.sin(2 * np.pi * 443 * t_short)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (data, freq_label) in zip(axes, [(data_440, "440Hz"), (data_443, "443Hz")]):
    mag = np.abs(np.fft.fft(data))
    freqs = np.fft.fftfreq(len(data), 1 / sample_rate)
    pos = freqs >= 0
    rng = freqs[pos] <= 600
    ax.plot(freqs[pos][rng], mag[pos][rng])
    ax.set_title(f"{freq_label}（窓関数なし）")
    ax.set_xlabel("周波数 (Hz)")
    ax.set_ylabel("振幅")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 窓関数で解決する

In [ ]:
# ハニング窓を適用
window = np.hanning(len(t_short))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (data, freq_label) in zip(axes, [(data_440, "440Hz"), (data_443, "443Hz")]):
    windowed = data * window
    mag = np.abs(np.fft.fft(windowed))
    freqs = np.fft.fftfreq(len(data), 1 / sample_rate)
    pos = freqs >= 0
    rng = freqs[pos] <= 600
    ax.plot(freqs[pos][rng], mag[pos][rng])
    ax.set_title(f"{freq_label}（ハニング窓あり）")
    ax.set_xlabel("周波数 (Hz)")
    ax.set_ylabel("振幅")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 窓関数の形

In [ ]:
# 代表的な窓関数の形
n = 256
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(np.ones(n), label="矩形窓（窓なし）", alpha=0.7)
ax.plot(np.hanning(n), label="ハニング窓", alpha=0.7)
ax.plot(np.hamming(n), label="ハミング窓", alpha=0.7)
ax.plot(np.blackman(n), label="ブラックマン窓", alpha=0.7)
ax.set_title("代表的な窓関数")
ax.set_xlabel("サンプル")
ax.set_ylabel("振幅")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 13.7 plot_spectrum() の中身を覗く

In [ ]:
def plot_spectrum(signal, max_freq=5000, title="周波数スペクトラム"):
    """plot_spectrum() の中身（簡略版）"""
    data = signal.data
    sr = signal.sample_rate

    # ① 窓関数を適用（スペクトル漏れを抑える）
    windowed = data * np.hanning(len(data))

    # ② FFT を実行
    fft_result = np.fft.fft(windowed)

    # ③ 振幅を取り出す（複素数 → 絶対値）
    fft_magnitude = np.abs(fft_result)

    # ④ 周波数軸を計算
    fft_freq = np.fft.fftfreq(len(data), 1 / sr)

    # ⑤ 正の周波数のみを取り出す
    positive_idx = fft_freq >= 0
    freq = fft_freq[positive_idx]
    magnitude = fft_magnitude[positive_idx]

    # ⑥ dB スケールに変換して表示
    range_idx = freq <= max_freq
    plt.plot(freq[range_idx], 20 * np.log10(magnitude[range_idx] + 1e-10))

### 上昇音で試してみよう

In [ ]:
# 周波数が 220Hz → 880Hz に上昇する音（チャープ信号）
dur = 2.0
t_chirp = np.linspace(0, dur, int(sample_rate * dur), endpoint=False)
freq_start, freq_end = 220, 880
instantaneous_freq = freq_start + (freq_end - freq_start) * t_chirp / dur
phase = 2 * np.pi * np.cumsum(instantaneous_freq) / sample_rate
chirp = np.sin(phase)

chirp_signal = AudioSignal(chirp, sample_rate)
display(play_sound(chirp_signal, "上昇音（220Hz → 880Hz）"))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# 波形
axes[0].plot(t_chirp[:4410], chirp[:4410])
axes[0].set_title("波形（先頭 0.1 秒）")
axes[0].set_xlabel("時間 (秒)")
axes[0].set_ylabel("振幅")
axes[0].grid(True, alpha=0.3)

# スペクトログラム
axes[1].specgram(chirp, Fs=sample_rate, NFFT=1024, noverlap=512, cmap='magma')
axes[1].set_title("スペクトログラム")
axes[1].set_xlabel("時間 (秒)")
axes[1].set_ylabel("周波数 (Hz)")
axes[1].set_ylim(0, 2000)

plt.tight_layout()
plt.show()

### ピアノ風の音

In [ ]:
f0 = 262  # C4
dur = 2.0

# ピアノ風の倍音構成
harmonics = {1: 1.0, 2: 0.7, 3: 0.3, 4: 0.2, 5: 0.1, 6: 0.05}
wave = additive_synth(f0, harmonics, duration=dur)

# エンベロープを適用
env = adsr(dur, attack=0.01, decay=0.4, sustain=0.3, release=0.5)
piano = AudioSignal(wave.data * env.data, sample_rate)
display(play_sound(piano, "ピアノ風の音"))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# スペクトル
plot_data = piano.data * np.hanning(len(piano.data))
fft_mag = np.abs(np.fft.fft(plot_data))
freqs = np.fft.fftfreq(len(plot_data), 1 / sample_rate)
pos = freqs >= 0
rng = freqs[pos] <= 3000

axes[0].plot(freqs[pos][rng], 20 * np.log10(fft_mag[pos][rng] + 1e-10))
axes[0].set_title("ピアノ風の音のスペクトル")
axes[0].set_xlabel("周波数 (Hz)")
axes[0].set_ylabel("振幅 (dB)")
axes[0].grid(True, alpha=0.3)

# スペクトログラム
axes[1].specgram(piano.data, Fs=sample_rate, NFFT=2048, noverlap=1536, cmap='magma')
axes[1].set_title("ピアノ風の音のスペクトログラム")
axes[1].set_xlabel("時間 (秒)")
axes[1].set_ylabel("周波数 (Hz)")
axes[1].set_ylim(0, 3000)

plt.tight_layout()
plt.show()

### 2つの音色を比較する

In [ ]:
dur = 1.5
sq = square_wave(f0, dur)
saw = sawtooth_wave(f0, dur)

# エンベロープ適用
env = adsr(dur, attack=0.01, decay=0.2, sustain=0.6, release=0.3)
sq_env = AudioSignal(sq.data * env.data, sample_rate)
saw_env = AudioSignal(saw.data * env.data, sample_rate)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].specgram(sq_env.data, Fs=sample_rate, NFFT=2048, noverlap=1536, cmap='magma')
axes[0].set_title("矩形波")
axes[0].set_ylabel("周波数 (Hz)")
axes[0].set_xlabel("時間 (秒)")
axes[0].set_ylim(0, 5000)

axes[1].specgram(saw_env.data, Fs=sample_rate, NFFT=2048, noverlap=1536, cmap='magma')
axes[1].set_title("ノコギリ波")
axes[1].set_ylabel("周波数 (Hz)")
axes[1].set_xlabel("時間 (秒)")
axes[1].set_ylim(0, 5000)

plt.tight_layout()
plt.show()

In [ ]:
display(play_sound(sq_env, "矩形波（エンベロープ付き）"))
display(play_sound(saw_env, "ノコギリ波（エンベロープ付き）"))

### メジャーコードとマイナーコード

In [ ]:
from audio_lib import note_to_frequency

# C メジャー: C4, E4, G4
c_major_freqs = [note_to_frequency(n) for n in [60, 64, 67]]

# C マイナー: C4, Eb4, G4
c_minor_freqs = [note_to_frequency(n) for n in [60, 63, 67]]

dur = 1.5

def make_chord(freqs, duration):
    """複数の周波数を同時に鳴らして和音を作る"""
    t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
    data = sum(np.sin(2 * np.pi * f * t) for f in freqs) / len(freqs)
    return AudioSignal(data, sample_rate)

major = make_chord(c_major_freqs, dur)
minor = make_chord(c_minor_freqs, dur)

display(play_sound(major, "Cメジャー（C4, E4, G4）"))
display(play_sound(minor, "Cマイナー（C4, Eb4, G4）"))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

for ax, (sig, name) in zip(axes, [(major, "Cメジャー"), (minor, "Cマイナー")]):
    windowed = sig.data * np.hanning(len(sig.data))
    mag = np.abs(np.fft.fft(windowed))
    freqs = np.fft.fftfreq(len(windowed), 1 / sample_rate)
    pos = freqs >= 0
    rng = freqs[pos] <= 1000

    ax.plot(freqs[pos][rng], mag[pos][rng])
    ax.set_ylabel(f"{name}\n振幅")
    ax.grid(True, alpha=0.3)

    # ピーク周波数をラベル表示
    for f in [note_to_frequency(n) for n in ([60, 64, 67] if "メジャー" in name else [60, 63, 67])]:
        ax.axvline(x=f, color='red', alpha=0.3, linestyle='--')

axes[-1].set_xlabel("周波数 (Hz)")
fig.suptitle("メジャーコードとマイナーコードのスペクトル", fontsize=14)
plt.tight_layout()
plt.show()

## 13.11 Lesson 9 との接続 — エフェクトを周波数で見る

In [ ]:
from audio_lib import Reverb as SimpleReverb

dry = AudioSignal(
    sawtooth_wave(440, 1.5).data * adsr(1.5, 0.01, 0.3, 0.5, 0.3).data,
    sample_rate,
)

reverb = SimpleReverb(room_size=0.8, damping=0.5, wet_level=0.5)
wet = reverb.process(dry)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (sig, name) in zip(axes, [(dry, "ドライ"), (wet, "リバーブあり")]):
    ax.specgram(sig.data, Fs=sample_rate, NFFT=2048, noverlap=1536, cmap='magma')
    ax.set_title(name)
    ax.set_xlabel("時間 (秒)")
    ax.set_ylabel("周波数 (Hz)")
    ax.set_ylim(0, 5000)

plt.tight_layout()
plt.show()